# Can politeness scores predict the winning CMV reply?

Evaluate the saved Phase 2 Polite Guard scores at 128, 256, and 510 tokens alongside word count and ConvoKit courtesy. Fit paired logistic regression to the signed A-minus-B feature differences. Use the frozen A/B orientation and the same three OP-discussion-grouped training folds for every model. Scaling is fit within each training fold. The published held-out data are untouched.

ROC-AUC and average precision summarize ranking across thresholds. Pairwise accuracy is the rate of picking the winning reply. Precision at 0.5 is the fraction of predicted A wins that actually went to A. Because every pair has exactly one winner and the classifier chooses one reply, precision across all chosen winners equals pairwise accuracy. Lower log loss indicates better probability estimates.

The standalone scores show how much signal each politeness measure carries on its own. The models with word count or word count and courtesy show whether contextual politeness adds predictive information beyond those inputs. Paired percentile intervals use the same 1,000 resamples of OP discussions and keep the out-of-fold predictions fixed.

[Evaluation runner](evaluate_politeness.py)

In [1]:
from pathlib import Path
import json

import pandas as pd

project = next(
    path / "project" for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "team-setup").is_dir()
)
run_id = "20261124T055238584162Z"
run = project / "data/processed/linguistic-features/politeness-experiment/phase-4" / run_id

results = pd.read_csv(run / "predictive_comparison.csv")
metadata = json.loads((run / "experiment_metadata.json").read_text())
assert metadata["split"] == "train"
order = results["model"].unique()
metrics = ["ROC-AUC", "Pairwise accuracy", "Precision at 0.5", "Average precision", "Log loss"]
print(f"{metadata['pairs']:,} pairs, {metadata['discussions']:,} discussions, {metadata['folds']} grouped folds")
print(f"A-wins prevalence {metadata['a_win_prevalence']:.1%}")
print(results.pivot(index="model", columns="metric", values="score").loc[order, metrics].round(4).to_string())


3,451 pairs, 2,507 discussions, 3 grouped folds
A-wins prevalence 49.7%
metric                                    ROC-AUC  Pairwise accuracy  Precision at 0.5  Average precision  Log loss
model                                                                                                              
Word count                                 0.6794             0.6259            0.6252             0.6684    0.6466
Courtesy count                             0.4915             0.5001            0.4689             0.4924    0.6935
Word count + courtesy                      0.6783             0.6256            0.6249             0.6670    0.6467
Polite Guard 128                           0.5799             0.5682            0.5768             0.5603    0.6867
Word count + Polite Guard 128              0.6826             0.6305            0.6300             0.6723    0.6431
Word count + courtesy + Polite Guard 128   0.6811             0.6294            0.6280             0.6718    0.6430


## Incremental performance

Word count is the reference for adding courtesy or contextual politeness alone. The combined models use word count plus courtesy as their reference.

In [2]:

fields = ["model", "reference", "metric", "difference", "difference_ci_low", "difference_ci_high"]
increment = results.loc[
    results["model"].str.startswith("Word count +") & results["metric"].isin(metrics[:-1]), fields
]
print(increment.round(4).to_string(index=False))


                                   model             reference            metric  difference  difference_ci_low  difference_ci_high
                   Word count + courtesy            Word count           ROC-AUC     -0.0011            -0.0043              0.0018
                   Word count + courtesy            Word count Average precision     -0.0014            -0.0050              0.0018
                   Word count + courtesy            Word count Pairwise accuracy     -0.0003            -0.0044              0.0040
                   Word count + courtesy            Word count  Precision at 0.5     -0.0004            -0.0049              0.0046
           Word count + Polite Guard 128            Word count           ROC-AUC      0.0032            -0.0047              0.0101
           Word count + Polite Guard 128            Word count Average precision      0.0039            -0.0034              0.0105
           Word count + Polite Guard 128            Word count Pairwise accu

## What the scores add so far

**The contextual scores carry a modest standalone signal.** The 128-token score reached 0.580 ROC-AUC and 56.8% pairwise accuracy, the highest standalone results among the three chunk sizes. The courtesy count reached 0.491 ROC-AUC and 50.0% pairwise accuracy.

**The gain beyond length was small and uncertain.** Word count alone reached 0.679 ROC-AUC and 62.6% accuracy. Adding the 128-token score gave the highest observed ROC-AUC (0.683), and adding the 256-token score gave the highest observed accuracy (63.3%) and precision at 0.5 (63.2%). The accuracy increase for 256 tokens was 0.75 percentage points, with a 95% interval of −0.20 to +1.73 points. For all three sizes, the intervals for ROC-AUC, accuracy, and precision changes included zero, including when starting from word count plus courtesy.

These are exploratory comparisons in the training data. Human agreement and boundary robustness remain part of the feature selection decision.

## Does contextual politeness add to P6?

Add one signed feature to the existing P6 specification. It is the difference between the two replies’ saved 256-token Polite Guard scores. The 256-token configuration is the one carried through the boundary-robustness experiment. All original P6 features, including net sentiment, are included. Train both models on the same three OP-discussion-grouped folds and verify that the baseline reproduces the archived P6 predictions.

The table reports the extension minus P6. The 95% intervals resample OP discussions with out-of-fold predictions held fixed. Precision at 0.5 refers to predicted A wins. [Evaluation runner](evaluate_p6_politeness.py)

In [3]:
p6_politeness_run_id = "20261124T060114192084Z"
p6_politeness_run = (
    project / "data/processed/linguistic-features/paired-models/p6_plus_contextual_politeness"
    / p6_politeness_run_id
)
p6_manifest = json.loads((p6_politeness_run / "results.json").read_text())
p6_comparison = pd.read_csv(p6_politeness_run / "predictive_comparison.csv")
assert p6_manifest["split"] == "train" and p6_manifest["p6_oof_reproduced"]
print(p6_comparison.round(5).to_string(index=False))
changes = p6_manifest["accuracy_transitions"]
print(f"Corrected picks: {changes['new_correct']}. New incorrect picks: {changes['new_incorrect']}.")


           metric      p6  p6_plus_politeness   change   ci_low  ci_high
          ROC-AUC 0.74681             0.74614 -0.00067 -0.00208  0.00079
Average precision 0.73410             0.73371 -0.00039 -0.00182  0.00096
Pairwise accuracy 0.68009             0.67806 -0.00203 -0.00585  0.00173
         Log loss 0.59657             0.59684  0.00027 -0.00077  0.00129
 Precision at 0.5 0.67962             0.67788 -0.00175 -0.00601  0.00245
Corrected picks: 20. New incorrect picks: 27.


Adding contextual politeness yielded no clear improvement over P6. ROC-AUC changed by −0.00067 and pairwise accuracy by −0.20 percentage points. Forty-seven picks changed, with 20 new correct picks and 27 new incorrect picks. Every metric’s change interval included zero. These estimates are from grouped training folds, and the human review is pending.